In [1]:
learning_events_df = spark.table(
    "demo.silver.learning_events"
)

practice_attempts_df = spark.table(
    "demo.silver.practice_attempts"
)

dim_learner_df = spark.table(
    "demo.gold.dim_learner"
)

dim_topic_df = spark.table(
    "demo.gold.dim_topic"
)

question_bank_df = spark.table(
    "demo.silver.question_bank"
)

print("Learning events:", learning_events_df.count())
print("Practice attempts:", practice_attempts_df.count())
print("Gold learners:", dim_learner_df.count())
print("Gold topics:", dim_topic_df.count())
print("Questions:", question_bank_df.count())

Learning events: 6
Practice attempts: 5
Gold learners: 3
Gold topics: 10
Questions: 5


In [2]:
practice_attempts_df.groupBy(
    "event_id"
).count().orderBy(
    "event_id"
).show()

+--------+-----+
|event_id|count|
+--------+-----+
|evt_0002|    2|
|evt_0004|    2|
|evt_0006|    1|
+--------+-----+



In [3]:
from pyspark.sql.functions import (
    col,
    when,
    concat_ws,
    sha2
)

fact_learning_interaction_df = (
    learning_events_df.alias("e")
    .join(
        dim_learner_df
        .select("user_id", "user_key")
        .alias("l"),
        col("e.user_id") == col("l.user_id"),
        "inner"
    )
    .select(
        sha2(
            concat_ws(
                "||",
                col("e.event_id"),
                col("e.user_id"),
                col("e.session_id")
            ),
            256
        ).alias("interaction_key"),

        col("e.event_id"),
        col("l.user_key").cast("int"),
        col("e.session_id"),

        col("e.event_time"),
        col("e.event_date"),
        col("e.event_hour"),

        col("e.event_type"),
        col("e.source_system"),

        when(
            col("e.event_type") == "practice_submitted",
            True
        ).otherwise(False).alias("is_practice_event")
    )
)

fact_learning_interaction_df.orderBy(
    "event_time"
).show(truncate=False)

+----------------------------------------------------------------+--------+--------+-----------+-------------------+----------+----------+-----------------------+-------------+-----------------+
|interaction_key                                                 |event_id|user_key|session_id |event_time         |event_date|event_hour|event_type             |source_system|is_practice_event|
+----------------------------------------------------------------+--------+--------+-----------+-------------------+----------+----------+-----------------------+-------------+-----------------+
|9d0d7dae535435bc477e8ca3cd26e26ecb16e7018bcaf4b401b6e3a07f2a6a70|evt_0001|1       |session_001|2026-07-20 09:00:05|2026-07-20|9         |ai_learning_interaction|chat         |false            |
|184efbe55fd066928e363a7b4f3c6a199fc3390d8abe66caf33509403e97acda|evt_0002|1       |session_001|2026-07-20 09:12:00|2026-07-20|9         |practice_submitted     |practice_app |true             |
|7b3847c0b72321e9e8232d65

In [4]:
spark.sql("""
DELETE FROM demo.gold.fact_learning_interaction
""")

DataFrame[]

In [5]:
fact_learning_interaction_df.writeTo(
    "demo.gold.fact_learning_interaction"
).append()

In [6]:
spark.sql("""
SELECT
    interaction_key,
    event_id,
    user_key,
    session_id,
    event_time,
    event_type,
    source_system,
    is_practice_event
FROM demo.gold.fact_learning_interaction
ORDER BY event_time
""").show(truncate=False)

+----------------------------------------------------------------+--------+--------+-----------+-------------------+-----------------------+-------------+-----------------+
|interaction_key                                                 |event_id|user_key|session_id |event_time         |event_type             |source_system|is_practice_event|
+----------------------------------------------------------------+--------+--------+-----------+-------------------+-----------------------+-------------+-----------------+
|9d0d7dae535435bc477e8ca3cd26e26ecb16e7018bcaf4b401b6e3a07f2a6a70|evt_0001|1       |session_001|2026-07-20 09:00:05|ai_learning_interaction|chat         |false            |
|184efbe55fd066928e363a7b4f3c6a199fc3390d8abe66caf33509403e97acda|evt_0002|1       |session_001|2026-07-20 09:12:00|practice_submitted     |practice_app |true             |
|7b3847c0b72321e9e8232d6577587e18f6e8b295457b20f3e750f649a0c46ec1|evt_0003|2       |session_002|2026-07-21 11:00:04|ai_learning_interac

In [7]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT interaction_key) AS distinct_interaction_keys,
    COUNT(DISTINCT event_id) AS distinct_event_ids
FROM demo.gold.fact_learning_interaction
""").show()

+----------+-------------------------+------------------+
|total_rows|distinct_interaction_keys|distinct_event_ids|
+----------+-------------------------+------------------+
|         6|                        6|                 6|
+----------+-------------------------+------------------+



In [8]:
from pyspark.sql.functions import (
    col,
    concat_ws,
    sha2,
    lower,
    trim
)

question_topics_df = (
    question_bank_df.alias("q")
    .join(
        dim_topic_df
        .filter(col("taxonomy_level") == "subtopic")
        .select(
            "topic_key",
            "normalized_topic_name"
        )
        .alias("t"),
        lower(trim(col("q.subtopic")))
        == col("t.normalized_topic_name"),
        "left"
    )
    .select(
        col("q.question_id"),
        col("q.question_version"),
        col("t.topic_key")
    )
)

fact_practice_attempt_df = (
    practice_attempts_df.alias("a")
    .join(
        dim_learner_df
        .select("user_id", "user_key")
        .alias("l"),
        col("a.user_id") == col("l.user_id"),
        "inner"
    )
    .join(
        question_topics_df.alias("q"),
        (
            (col("a.question_id") == col("q.question_id"))
            & (
                col("a.question_version")
                == col("q.question_version")
            )
        ),
        "left"
    )
    .select(
        sha2(
            concat_ws(
                "||",
                col("a.attempt_id"),
                col("a.event_id"),
                col("a.question_id")
            ),
            256
        ).alias("attempt_key"),

        col("a.attempt_id"),
        col("a.event_id"),
        col("l.user_key").cast("int"),
        col("a.session_id"),
        col("q.topic_key").cast("int"),

        col("a.practice_id"),
        col("a.question_id"),
        col("a.question_version").cast("int"),
        col("a.attempt_time"),

        col("a.selected_option_letter"),
        col("a.is_correct"),
        col("a.score").cast("float"),
        col("a.hints_used").cast("int"),
        col("a.attempt_duration_seconds").cast("int"),
        col("a.attempt_number").cast("int")
    )
)

fact_practice_attempt_df.orderBy(
    "attempt_time",
    "attempt_id"
).show(truncate=False)

+----------------------------------------------------------------+----------------------------------------------------------------+--------+--------+-----------+---------+------------+------------+----------------+-------------------+----------------------+----------+-----+----------+------------------------+--------------+
|attempt_key                                                     |attempt_id                                                      |event_id|user_key|session_id |topic_key|practice_id |question_id |question_version|attempt_time       |selected_option_letter|is_correct|score|hints_used|attempt_duration_seconds|attempt_number|
+----------------------------------------------------------------+----------------------------------------------------------------+--------+--------+-----------+---------+------------+------------+----------------+-------------------+----------------------+----------+-----+----------+------------------------+--------------+
|4f8e99730047d69c06e41

In [9]:
spark.sql("""
DELETE FROM demo.gold.fact_practice_attempt
""")

DataFrame[]

In [10]:
fact_practice_attempt_df.writeTo(
    "demo.gold.fact_practice_attempt"
).append()

In [11]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT attempt_key) AS distinct_attempt_keys,
    COUNT(DISTINCT attempt_id) AS distinct_attempt_ids,
    COUNT(DISTINCT event_id) AS distinct_event_ids
FROM demo.gold.fact_practice_attempt
""").show()

+----------+---------------------+--------------------+------------------+
|total_rows|distinct_attempt_keys|distinct_attempt_ids|distinct_event_ids|
+----------+---------------------+--------------------+------------------+
|         5|                    5|                   5|                 3|
+----------+---------------------+--------------------+------------------+



In [12]:
from pyspark.sql.functions import (
    col,
    min,
    max,
    count,
    first,
    unix_timestamp,
    round
)

session_events_df = (
    fact_learning_interaction_df
    .groupBy(
        "session_id",
        "user_key"
    )
    .agg(
        min("event_time").alias("session_start_time"),
        max("event_time").alias("session_end_time"),
        count("*").cast("int").alias("total_events"),
        first("source_system").alias("main_source_system")
    )
    .withColumn(
        "session_duration_minutes",
        round(
            (
                unix_timestamp("session_end_time")
                - unix_timestamp("session_start_time")
            ) / 60
        ).cast("int")
    )
)

session_events_df.orderBy(
    "session_id"
).show(truncate=False)

+-----------+--------+-------------------+-------------------+------------+------------------+------------------------+
|session_id |user_key|session_start_time |session_end_time   |total_events|main_source_system|session_duration_minutes|
+-----------+--------+-------------------+-------------------+------------+------------------+------------------------+
|session_001|1       |2026-07-20 09:00:05|2026-07-20 09:12:00|2           |chat              |12                      |
|session_002|2       |2026-07-21 11:00:04|2026-07-21 11:11:00|2           |chat              |11                      |
|session_003|3       |2026-07-22 14:00:05|2026-07-22 14:08:00|2           |chat              |8                       |
+-----------+--------+-------------------+-------------------+------------+------------------+------------------------+



In [13]:
from pyspark.sql.functions import (
    count,
    sum,
    avg
)

session_attempts_df = (
    fact_practice_attempt_df
    .groupBy(
        "session_id",
        "user_key"
    )
    .agg(
        count("*").cast("int").alias("total_attempts"),
        sum("hints_used").cast("int").alias("total_hints_used"),
        avg("score").cast("float").alias("average_practice_score")
    )
)

session_attempts_df.orderBy(
    "session_id"
).show(truncate=False)

+-----------+--------+--------------+----------------+----------------------+
|session_id |user_key|total_attempts|total_hints_used|average_practice_score|
+-----------+--------+--------------+----------------+----------------------+
|session_001|1       |2             |1               |0.5                   |
|session_002|2       |2             |0               |0.5                   |
|session_003|3       |1             |0               |1.0                   |
+-----------+--------+--------------+----------------+----------------------+



In [14]:
from pyspark.sql.functions import col, coalesce, lit, when

fact_learning_session_df = (
    session_events_df.alias("e")
    .join(
        session_attempts_df.alias("a"),
        (
            (col("e.session_id") == col("a.session_id"))
            & (col("e.user_key") == col("a.user_key"))
        ),
        "left"
    )
    .select(
        col("e.session_id"),
        col("e.user_key").cast("int"),

        col("e.session_start_time"),
        col("e.session_end_time"),
        col("e.session_duration_minutes").cast("int"),

        col("e.main_source_system"),

        col("e.total_events").cast("int"),

        coalesce(
            col("a.total_attempts"),
            lit(0)
        ).cast("int").alias("total_attempts"),

        coalesce(
            col("a.total_hints_used"),
            lit(0)
        ).cast("int").alias("total_hints_used"),

        col("a.average_practice_score").cast("float"),

        when(
            col("a.total_attempts") > 0,
            lit("completed")
        )
        .otherwise(
            lit("interrupted")
        )
        .alias("session_status")
    )
)

fact_learning_session_df.orderBy(
    "session_id"
).show(truncate=False)

+-----------+--------+-------------------+-------------------+------------------------+------------------+------------+--------------+----------------+----------------------+--------------+
|session_id |user_key|session_start_time |session_end_time   |session_duration_minutes|main_source_system|total_events|total_attempts|total_hints_used|average_practice_score|session_status|
+-----------+--------+-------------------+-------------------+------------------------+------------------+------------+--------------+----------------+----------------------+--------------+
|session_001|1       |2026-07-20 09:00:05|2026-07-20 09:12:00|12                      |chat              |2           |2             |1               |0.5                   |completed     |
|session_002|2       |2026-07-21 11:00:04|2026-07-21 11:11:00|11                      |chat              |2           |2             |0               |0.5                   |completed     |
|session_003|3       |2026-07-22 14:00:05|2026-07-

In [15]:
spark.sql("""
DELETE FROM demo.gold.fact_learning_session
""")

DataFrame[]

In [16]:
fact_learning_session_df.writeTo(
    "demo.gold.fact_learning_session"
).append()

In [17]:
spark.sql("""
SELECT *
FROM demo.gold.fact_learning_session
ORDER BY session_id
""").show(truncate=False)

+-----------+--------+-------------------+-------------------+------------------------+------------------+------------+--------------+----------------+----------------------+--------------+
|session_id |user_key|session_start_time |session_end_time   |session_duration_minutes|main_source_system|total_events|total_attempts|total_hints_used|average_practice_score|session_status|
+-----------+--------+-------------------+-------------------+------------------------+------------------+------------+--------------+----------------+----------------------+--------------+
|session_001|1       |2026-07-20 09:00:05|2026-07-20 09:12:00|12                      |chat              |2           |2             |1               |0.5                   |completed     |
|session_002|2       |2026-07-21 11:00:04|2026-07-21 11:11:00|11                      |chat              |2           |2             |0               |0.5                   |completed     |
|session_003|3       |2026-07-22 14:00:05|2026-07-

In [18]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT session_id) AS distinct_session_ids
FROM demo.gold.fact_learning_session
""").show()

+----------+--------------------+
|total_rows|distinct_session_ids|
+----------+--------------------+
|         3|                   3|
+----------+--------------------+



In [19]:
pre_feedback_df = spark.table(
    "demo.silver.pre_practice_feedback"
)

post_feedback_df = spark.table(
    "demo.silver.post_practice_feedback"
)

check_in_df = spark.table(
    "demo.silver.learner_check_in"
)

check_in_topics_df = spark.table(
    "demo.silver.learner_check_in_topics"
)

print("Pre-practice feedback:", pre_feedback_df.count())
print("Post-practice feedback:", post_feedback_df.count())
print("Check-in parent rows:", check_in_df.count())
print("Check-in topic rows:", check_in_topics_df.count())

Pre-practice feedback: 3
Post-practice feedback: 3
Check-in parent rows: 3
Check-in topic rows: 5


In [20]:
from pyspark.sql.functions import (
    col,
    lit,
    sha2,
    concat_ws
)

pre_feedback_fact_df = (
    pre_feedback_df.alias("f")
    .join(
        dim_learner_df
        .select("user_id", "user_key")
        .alias("l"),
        col("f.user_id") == col("l.user_id"),
        "inner"
    )
    .select(
        sha2(
            concat_ws(
                "||",
                lit("before_practice"),
                col("f.feedback_id"),
                col("f.user_id")
            ),
            256
        ).alias("feedback_key"),

        col("l.user_key").cast("int"),
        col("f.session_id"),
        lit(None).cast("int").alias("topic_key"),
        col("f.practice_id"),

        lit("before_practice").alias("feedback_stage"),

        col("f.feedback_time"),
        col("f.ingestion_time"),
        col("f.delay_minutes").cast("int"),

        col("f.confidence_before_score")
            .cast("int")
            .alias("confidence_score"),

        col("f.perceived_understanding_before_score")
            .cast("int")
            .alias("perceived_understanding_score"),

        col("f.expected_difficulty_score")
            .cast("int")
            .alias("perceived_difficulty_score"),

        lit(None).cast("int").alias("motivation_score"),
        lit(None).cast("int").alias("stress_score"),
        lit(None).cast("boolean").alias("still_confused")
    )
)

pre_feedback_fact_df.orderBy(
    "feedback_time"
).show(truncate=False)

+----------------------------------------------------------------+--------+-----------+---------+------------+---------------+-------------------+-------------------+-------------+----------------+-----------------------------+--------------------------+----------------+------------+--------------+
|feedback_key                                                    |user_key|session_id |topic_key|practice_id |feedback_stage |feedback_time      |ingestion_time     |delay_minutes|confidence_score|perceived_understanding_score|perceived_difficulty_score|motivation_score|stress_score|still_confused|
+----------------------------------------------------------------+--------+-----------+---------+------------+---------------+-------------------+-------------------+-------------+----------------+-----------------------------+--------------------------+----------------+------------+--------------+
|87109ded9e6bea4f91dd915e10e54887dc7eec3787019c0bbe20f7ab9f02d435|1       |session_001|NULL     |pra

In [21]:
post_feedback_fact_df = (
    post_feedback_df.alias("f")
    .join(
        dim_learner_df
        .select("user_id", "user_key")
        .alias("l"),
        col("f.user_id") == col("l.user_id"),
        "inner"
    )
    .select(
        sha2(
            concat_ws(
                "||",
                lit("after_practice"),
                col("f.feedback_id"),
                col("f.user_id")
            ),
            256
        ).alias("feedback_key"),

        col("l.user_key").cast("int"),
        col("f.session_id"),
        lit(None).cast("int").alias("topic_key"),
        col("f.practice_id"),

        lit("after_practice").alias("feedback_stage"),

        col("f.feedback_time"),
        col("f.ingestion_time"),
        col("f.delay_minutes").cast("int"),

        col("f.confidence_after_score")
            .cast("int")
            .alias("confidence_score"),

        col("f.perceived_understanding_after_score")
            .cast("int")
            .alias("perceived_understanding_score"),

        col("f.perceived_difficulty_score")
            .cast("int")
            .alias("perceived_difficulty_score"),

        lit(None).cast("int").alias("motivation_score"),
        lit(None).cast("int").alias("stress_score"),

        col("f.still_confused")
            .cast("boolean")
            .alias("still_confused")
    )
)

post_feedback_fact_df.orderBy(
    "feedback_time"
).show(truncate=False)

+----------------------------------------------------------------+--------+-----------+---------+------------+--------------+-------------------+-------------------+-------------+----------------+-----------------------------+--------------------------+----------------+------------+--------------+
|feedback_key                                                    |user_key|session_id |topic_key|practice_id |feedback_stage|feedback_time      |ingestion_time     |delay_minutes|confidence_score|perceived_understanding_score|perceived_difficulty_score|motivation_score|stress_score|still_confused|
+----------------------------------------------------------------+--------+-----------+---------+------------+--------------+-------------------+-------------------+-------------+----------------+-----------------------------+--------------------------+----------------+------------+--------------+
|e218f89800949134b7eaabd6bd501a4fe8dd535813937c036ce54d437da34082|1       |session_001|NULL     |practi

In [22]:
check_in_feedback_fact_df = (
    check_in_topics_df.alias("t")
    .join(
        check_in_df.alias("c"),
        col("t.feedback_id") == col("c.feedback_id"),
        "inner"
    )
    .join(
        dim_learner_df
        .select("user_id", "user_key")
        .alias("l"),
        col("t.user_id") == col("l.user_id"),
        "inner"
    )
    .join(
        dim_topic_df
        .select(
            "topic_key",
            "topic_id"
        )
        .alias("d"),
        col("t.topic_id") == col("d.topic_id"),
        "left"
    )
    .select(
        sha2(
            concat_ws(
                "||",
                lit("general_check_in"),
                col("t.feedback_id"),
                col("t.topic_id"),
                col("t.user_id")
            ),
            256
        ).alias("feedback_key"),

        col("l.user_key").cast("int"),
        col("t.session_id"),
        col("d.topic_key").cast("int"),
        lit(None).cast("string").alias("practice_id"),

        lit("general_check_in").alias("feedback_stage"),

        col("t.feedback_time"),
        col("c.ingestion_time"),
        col("c.delay_minutes").cast("int"),

        col("t.topic_confidence_score")
            .cast("int")
            .alias("confidence_score"),

        col("t.perceived_understanding_score")
            .cast("int")
            .alias("perceived_understanding_score"),

        lit(None)
            .cast("int")
            .alias("perceived_difficulty_score"),

        col("c.overall_motivation_score")
            .cast("int")
            .alias("motivation_score"),

        col("c.overall_stress_score")
            .cast("int")
            .alias("stress_score"),

        col("t.still_confused")
            .cast("boolean")
            .alias("still_confused")
    )
)

check_in_feedback_fact_df.orderBy(
    "feedback_time",
    "topic_key"
).show(truncate=False)

+----------------------------------------------------------------+--------+-----------+---------+-----------+----------------+-------------------+-------------------+-------------+----------------+-----------------------------+--------------------------+----------------+------------+--------------+
|feedback_key                                                    |user_key|session_id |topic_key|practice_id|feedback_stage  |feedback_time      |ingestion_time     |delay_minutes|confidence_score|perceived_understanding_score|perceived_difficulty_score|motivation_score|stress_score|still_confused|
+----------------------------------------------------------------+--------+-----------+---------+-----------+----------------+-------------------+-------------------+-------------+----------------+-----------------------------+--------------------------+----------------+------------+--------------+
|82fbfa584e59c11a8f493e84b02debe3869d0d51d819eadad2497558c65c4b5d|1       |session_001|NULL     |NUL

In [23]:
from pyspark.sql.functions import (
    col,
    lit,
    sha2,
    concat_ws,
    regexp_replace,
    lower,
    trim
)

check_in_topics_normalized_df = (
    check_in_topics_df
    .withColumn(
        "normalized_check_in_topic",
        lower(
            trim(
                regexp_replace(
                    regexp_replace(
                        col("topic_id"),
                        "^topic_",
                        ""
                    ),
                    "_",
                    " "
                )
            )
        )
    )
)

dim_topic_lookup_df = (
    dim_topic_df
    .filter(
        col("taxonomy_level").isin(
            "topic",
            "subtopic",
            "concept"
        )
    )
    .select(
        "topic_key",
        "topic_id",
        "normalized_topic_name",
        "taxonomy_level",
        "validation_status"
    )
)

check_in_feedback_fact_df = (
    check_in_topics_normalized_df.alias("t")
    .join(
        check_in_df.alias("c"),
        col("t.feedback_id") == col("c.feedback_id"),
        "inner"
    )
    .join(
        dim_learner_df
        .select("user_id", "user_key")
        .alias("l"),
        col("t.user_id") == col("l.user_id"),
        "inner"
    )
    .join(
        dim_topic_lookup_df.alias("d"),
        col("t.normalized_check_in_topic")
        == col("d.normalized_topic_name"),
        "left"
    )
    .select(
        sha2(
            concat_ws(
                "||",
                lit("general_check_in"),
                col("t.feedback_id"),
                col("t.topic_id"),
                col("t.user_id")
            ),
            256
        ).alias("feedback_key"),

        col("l.user_key").cast("int"),
        col("t.session_id"),
        col("d.topic_key").cast("int"),
        lit(None).cast("string").alias("practice_id"),

        lit("general_check_in").alias("feedback_stage"),

        col("t.feedback_time"),
        col("c.ingestion_time"),
        col("c.delay_minutes").cast("int"),

        col("t.topic_confidence_score")
        .cast("int")
        .alias("confidence_score"),

        col("t.perceived_understanding_score")
        .cast("int")
        .alias("perceived_understanding_score"),

        lit(None)
        .cast("int")
        .alias("perceived_difficulty_score"),

        col("c.overall_motivation_score")
        .cast("int")
        .alias("motivation_score"),

        col("c.overall_stress_score")
        .cast("int")
        .alias("stress_score"),

        col("t.still_confused")
        .cast("boolean")
        .alias("still_confused")
    )
)

check_in_feedback_fact_df.orderBy(
    "feedback_time",
    "topic_key"
).show(truncate=False)

+----------------------------------------------------------------+--------+-----------+---------+-----------+----------------+-------------------+-------------------+-------------+----------------+-----------------------------+--------------------------+----------------+------------+--------------+
|feedback_key                                                    |user_key|session_id |topic_key|practice_id|feedback_stage  |feedback_time      |ingestion_time     |delay_minutes|confidence_score|perceived_understanding_score|perceived_difficulty_score|motivation_score|stress_score|still_confused|
+----------------------------------------------------------------+--------+-----------+---------+-----------+----------------+-------------------+-------------------+-------------+----------------+-----------------------------+--------------------------+----------------+------------+--------------+
|82fbfa584e59c11a8f493e84b02debe3869d0d51d819eadad2497558c65c4b5d|1       |session_001|5        |NUL

In [24]:
fact_learning_feedback_df = (
    pre_feedback_fact_df
    .unionByName(post_feedback_fact_df)
    .unionByName(check_in_feedback_fact_df)
)

In [25]:
fact_learning_feedback_df.groupBy(
    "feedback_stage"
).count().orderBy(
    "feedback_stage"
).show()

+----------------+-----+
|  feedback_stage|count|
+----------------+-----+
|  after_practice|    3|
| before_practice|    3|
|general_check_in|    5|
+----------------+-----+



In [26]:
print(
    "Total feedback fact rows:",
    fact_learning_feedback_df.count()
)

print(
    "Distinct feedback keys:",
    fact_learning_feedback_df
    .select("feedback_key")
    .distinct()
    .count()
)

Total feedback fact rows: 11
Distinct feedback keys: 11


In [27]:
spark.sql("""
DELETE FROM demo.gold.fact_learning_feedback
""")

DataFrame[]

In [28]:
fact_learning_feedback_df.writeTo(
    "demo.gold.fact_learning_feedback"
).append()

In [29]:
spark.sql("""
SELECT
    feedback_key,
    user_key,
    session_id,
    topic_key,
    practice_id,
    feedback_stage,
    feedback_time,
    confidence_score,
    perceived_understanding_score,
    perceived_difficulty_score,
    motivation_score,
    stress_score,
    still_confused
FROM demo.gold.fact_learning_feedback
ORDER BY feedback_time, feedback_stage, topic_key
""").show(truncate=False)

+----------------------------------------------------------------+--------+-----------+---------+------------+----------------+-------------------+----------------+-----------------------------+--------------------------+----------------+------------+--------------+
|feedback_key                                                    |user_key|session_id |topic_key|practice_id |feedback_stage  |feedback_time      |confidence_score|perceived_understanding_score|perceived_difficulty_score|motivation_score|stress_score|still_confused|
+----------------------------------------------------------------+--------+-----------+---------+------------+----------------+-------------------+----------------+-----------------------------+--------------------------+----------------+------------+--------------+
|87109ded9e6bea4f91dd915e10e54887dc7eec3787019c0bbe20f7ab9f02d435|1       |session_001|NULL     |practice_001|before_practice |2026-07-20 09:04:00|4               |3                            |8    

In [30]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT feedback_key) AS distinct_feedback_keys
FROM demo.gold.fact_learning_feedback
""").show()

+----------+----------------------+
|total_rows|distinct_feedback_keys|
+----------+----------------------+
|        11|                    11|
+----------+----------------------+



In [31]:
validated_insights_df = spark.table(
    "demo.silver.validated_learning_insights"
)

ai_insights_df = spark.table(
    "demo.silver.ai_extracted_insights"
)

dim_reference_source_df = spark.table(
    "demo.gold.dim_reference_source"
)

print(
    "Validated insight rows:",
    validated_insights_df.count()
)

print(
    "AI insight rows:",
    ai_insights_df.count()
)

print(
    "Gold reference rows:",
    dim_reference_source_df.count()
)

Validated insight rows: 9
AI insight rows: 9
Gold reference rows: 6


In [32]:
validated_insights_df.printSchema()

validated_insights_df.show(
    truncate=False
)

root
 |-- validation_id: string (nullable = true)
 |-- insight_id: string (nullable = true)
 |-- reference_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- dynamic_concept_name: string (nullable = true)
 |-- validation_time: timestamp (nullable = true)
 |-- semantic_match_score: float (nullable = true)
 |-- reliability_score: float (nullable = true)
 |-- contradiction_flag: boolean (nullable = true)
 |-- validation_notes: string (nullable = true)
 |-- validation_status: string (nullable = true)

+----------------------------------------------------------------+----------------------------------------------------------------+-------------+--------+-----------+--------------------+--------------------------+--------------------+-----------------+------------------+--------------------------------------------------------------------------------------------------------------------+-----------------+
|validation_id           

In [33]:
from pyspark.sql.functions import (
    col,
    lower,
    trim,
    sha2,
    concat_ws,
    lit
)

topic_validation_lookup_df = (
    dim_topic_df
    .filter(
        col("taxonomy_level").isin(
            "topic",
            "subtopic",
            "concept"
        )
    )
    .select(
        "topic_key",
        "normalized_topic_name"
    )
)

fact_ai_insight_validation_df = (
    validated_insights_df.alias("v")

    # מוסיף event_id, extracted_at ו-extraction_confidence
    .join(
        ai_insights_df.alias("a"),
        col("v.insight_id") == col("a.insight_id"),
        "inner"
    )

    # ממפה user_id ל-user_key
    .join(
        dim_learner_df
        .select("user_id", "user_key")
        .alias("l"),
        col("v.user_id") == col("l.user_id"),
        "inner"
    )

    # ממפה את שם המושג ל-topic_key
    .join(
        topic_validation_lookup_df.alias("t"),
        lower(trim(col("v.dynamic_concept_name")))
        == col("t.normalized_topic_name"),
        "left"
    )

    # ממפה reference_id ל-reference_key
    .join(
        dim_reference_source_df
        .select("reference_id", "reference_key")
        .alias("r"),
        col("v.reference_id") == col("r.reference_id"),
        "left"
    )

    .select(
        sha2(
            concat_ws(
                "||",
                col("v.validation_id"),
                col("v.insight_id"),
                col("v.reference_id")
            ),
            256
        ).alias("validation_key"),

        col("v.validation_id"),
        col("v.insight_id"),
        col("a.event_id"),

        col("l.user_key").cast("int"),
        col("v.session_id"),

        col("t.topic_key").cast("int"),
        col("r.reference_key").cast("int"),

        col("v.dynamic_concept_name"),

        col("a.extracted_at"),
        col("v.validation_time"),

        col("a.extraction_confidence").cast("float"),
        col("v.semantic_match_score").cast("float"),
        col("v.reliability_score").cast("float"),

        col("v.contradiction_flag").cast("boolean"),
        col("v.validation_status"),

        lit(1).cast("int").alias("match_rank")
    )
)

fact_ai_insight_validation_df.orderBy(
    "event_id",
    "dynamic_concept_name"
).show(truncate=False)

+----------------------------------------------------------------+----------------------------------------------------------------+----------------------------------------------------------------+--------+--------+-----------+---------+-------------+--------------------+-------------------+--------------------------+---------------------+--------------------+-----------------+------------------+-----------------+----------+
|validation_key                                                  |validation_id                                                   |insight_id                                                      |event_id|user_key|session_id |topic_key|reference_key|dynamic_concept_name|extracted_at       |validation_time           |extraction_confidence|semantic_match_score|reliability_score|contradiction_flag|validation_status|match_rank|
+----------------------------------------------------------------+----------------------------------------------------------------+-------------

In [34]:
from pyspark.sql.functions import col

fact_ai_insight_validation_df.filter(
    col("user_key").isNull()
    | col("topic_key").isNull()
    | col("reference_key").isNull()
    | col("event_id").isNull()
).show(truncate=False)

print(
    "Rows with missing Gold links:",
    fact_ai_insight_validation_df.filter(
        col("user_key").isNull()
        | col("topic_key").isNull()
        | col("reference_key").isNull()
        | col("event_id").isNull()
    ).count()
)

+--------------+-------------+----------+--------+--------+----------+---------+-------------+--------------------+------------+---------------+---------------------+--------------------+-----------------+------------------+-----------------+----------+
|validation_key|validation_id|insight_id|event_id|user_key|session_id|topic_key|reference_key|dynamic_concept_name|extracted_at|validation_time|extraction_confidence|semantic_match_score|reliability_score|contradiction_flag|validation_status|match_rank|
+--------------+-------------+----------+--------+--------+----------+---------+-------------+--------------------+------------+---------------+---------------------+--------------------+-----------------+------------------+-----------------+----------+
+--------------+-------------+----------+--------+--------+----------+---------+-------------+--------------------+------------+---------------+---------------------+--------------------+-----------------+------------------+--------------

In [35]:
spark.sql("""
DELETE FROM demo.gold.fact_ai_insight_validation
""")

DataFrame[]

In [36]:
fact_ai_insight_validation_df.writeTo(
    "demo.gold.fact_ai_insight_validation"
).append()

In [37]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT validation_key) AS distinct_validation_keys,
    COUNT(DISTINCT validation_id) AS distinct_validation_ids,
    COUNT(DISTINCT insight_id) AS distinct_insight_ids
FROM demo.gold.fact_ai_insight_validation
""").show()

+----------+------------------------+-----------------------+--------------------+
|total_rows|distinct_validation_keys|distinct_validation_ids|distinct_insight_ids|
+----------+------------------------+-----------------------+--------------------+
|         9|                       9|                      9|                   9|
+----------+------------------------+-----------------------+--------------------+



In [38]:
learner_concept_evidence_df = spark.table(
    "demo.silver.learner_concept_evidence"
)

print(
    "Learner concept evidence rows:",
    learner_concept_evidence_df.count()
)

learner_concept_evidence_df.printSchema()

learner_concept_evidence_df.orderBy(
    "user_id",
    "taxonomy_id",
    "evidence_time"
).show(truncate=False)

Learner concept evidence rows: 34
root
 |-- evidence_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- taxonomy_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- attempt_id: string (nullable = true)
 |-- feedback_id: string (nullable = true)
 |-- insight_id: string (nullable = true)
 |-- validation_id: string (nullable = true)
 |-- evidence_type: string (nullable = true)
 |-- evidence_time: timestamp (nullable = true)
 |-- is_correct: boolean (nullable = true)
 |-- score: float (nullable = true)
 |-- hints_used: integer (nullable = true)
 |-- attempt_duration_seconds: integer (nullable = true)
 |-- attempt_number: integer (nullable = true)
 |-- extraction_confidence: float (nullable = true)
 |-- semantic_match_score: float (nullable = true)
 |-- reliability_score: float (nullable = true)
 |-- contradiction_flag: boolean (nullable = true)
 |-- confidence_score: integer (nullable = true)
 |-- perceiv

26/07/29 06:55:59 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------------------------------------------------------------+--------+-----------+----------------------------------------------------------------+--------+----------------------------------------------------------------+------------+----------------------------------------------------------------+----------------------------------------------------------------+-----------------+--------------------------+----------+-----+----------+------------------------+--------------+---------------------+--------------------+-----------------+------------------+----------------+-----------------------------+--------------------------+--------------+----------------------------------+--------------------------+
|evidence_id                                                     |user_id |session_id |taxonomy_id                                                     |event_id|attempt_id                                                      |feedback_id |insight_id                                     

In [39]:
evidence_summary_df = (
    learner_concept_evidence_df
    .groupBy(
        "user_id",
        "taxonomy_id",
        "evidence_type"
    )
    .count()
    .orderBy(
        "user_id",
        "taxonomy_id",
        "evidence_type"
    )
)

evidence_summary_df.show(
    100,
    truncate=False
)

+--------+----------------------------------------------------------------+-----------------+-----+
|user_id |taxonomy_id                                                     |evidence_type    |count|
+--------+----------------------------------------------------------------+-----------------+-----+
|user_001|6735221ef265cb6c5eca0ce35f3502dfe2a10d9deea57eb0bb029627d6edd02f|ai_insight       |1    |
|user_001|6735221ef265cb6c5eca0ce35f3502dfe2a10d9deea57eb0bb029627d6edd02f|check_in         |1    |
|user_001|6735221ef265cb6c5eca0ce35f3502dfe2a10d9deea57eb0bb029627d6edd02f|post_feedback    |1    |
|user_001|6735221ef265cb6c5eca0ce35f3502dfe2a10d9deea57eb0bb029627d6edd02f|practice_attempt |2    |
|user_001|6735221ef265cb6c5eca0ce35f3502dfe2a10d9deea57eb0bb029627d6edd02f|pre_feedback     |1    |
|user_001|6735221ef265cb6c5eca0ce35f3502dfe2a10d9deea57eb0bb029627d6edd02f|validated_insight|1    |
|user_001|74aeea96e4ef671a43f520e045210290ff021dbd20708f4f12ee57573f855e54|ai_insight       |1    |


In [40]:
learner_concept_groups_df = (
    learner_concept_evidence_df
    .groupBy(
        "user_id",
        "taxonomy_id"
    )
    .agg(
        count("*").cast("int").alias("evidence_count"),
        min("evidence_time").alias("first_evidence_time"),
        max("evidence_time").alias("last_evidence_time")
    )
    .orderBy(
        "user_id",
        "taxonomy_id"
    )
)

learner_concept_groups_df.show(
    truncate=False
)

print(
    "Learner-concept groups:",
    learner_concept_groups_df.count()
)

+--------+----------------------------------------------------------------+--------------+-------------------+--------------------------+
|user_id |taxonomy_id                                                     |evidence_count|first_evidence_time|last_evidence_time        |
+--------+----------------------------------------------------------------+--------------+-------------------+--------------------------+
|user_001|6735221ef265cb6c5eca0ce35f3502dfe2a10d9deea57eb0bb029627d6edd02f|7             |2026-07-20 09:00:08|2026-07-28 09:51:34.380674|
|user_001|74aeea96e4ef671a43f520e045210290ff021dbd20708f4f12ee57573f855e54|2             |2026-07-20 09:00:08|2026-07-28 09:51:34.380674|
|user_001|da4bbec24d294be7e79f3037d748b8fb5c89cf0c7872a6a9b5f63b0169afb058|1             |2026-07-23 18:00:00|2026-07-23 18:00:00       |
|user_001|e3a962692f730e9c99e3c3950032800398d2bb5a3b08ed13020b599dd9b8a80a|2             |2026-07-20 09:00:08|2026-07-28 09:51:34.380674|
|user_002|3b4c81d554cb203b67ef6add

In [41]:
from pyspark.sql.functions import (
    col,
    when,
    avg,
    sum,
    count,
    max
)

learner_concept_components_df = (
    learner_concept_evidence_df
    .groupBy(
        "user_id",
        "taxonomy_id"
    )
    .agg(
        count("*")
            .cast("int")
            .alias("evidence_count"),

        max("evidence_time")
            .alias("last_evidence_time"),

        # ביצועי תרגול
        avg(
            when(
                col("evidence_type") == "practice_attempt",
                col("score")
            )
        )
        .cast("float")
        .alias("avg_practice_score"),

        sum(
            when(
                (col("evidence_type") == "practice_attempt")
                & (col("is_correct") == False),
                1
            ).otherwise(0)
        )
        .cast("int")
        .alias("incorrect_attempt_count"),

        # דיווח עצמי של התלמיד — המרה מסולם 1–10 ל־0–1 תתבצע בהמשך
        avg(
            when(
                col("confidence_score").isNotNull(),
                col("confidence_score")
            )
        )
        .cast("float")
        .alias("avg_confidence_raw"),

        avg(
            when(
                col("perceived_understanding_score").isNotNull(),
                col("perceived_understanding_score")
            )
        )
        .cast("float")
        .alias("avg_understanding_raw"),

        avg(
            when(
                col("perceived_difficulty_score").isNotNull(),
                col("perceived_difficulty_score")
            )
        )
        .cast("float")
        .alias("avg_difficulty_raw"),

        avg(
            when(
                col("evidence_type") == "validated_insight",
                col("reliability_score")
            )
        )
        .cast("float")
        .alias("avg_validated_reliability"),

        max(
            when(
                col("still_confused") == True,
                1
            ).otherwise(0)
        )
        .cast("int")
        .alias("has_confusion_signal")
    )
)

learner_concept_components_df.orderBy(
    "user_id",
    "taxonomy_id"
).show(truncate=False)

+--------+----------------------------------------------------------------+--------------+--------------------------+------------------+-----------------------+------------------+---------------------+------------------+-------------------------+--------------------+
|user_id |taxonomy_id                                                     |evidence_count|last_evidence_time        |avg_practice_score|incorrect_attempt_count|avg_confidence_raw|avg_understanding_raw|avg_difficulty_raw|avg_validated_reliability|has_confusion_signal|
+--------+----------------------------------------------------------------+--------------+--------------------------+------------------+-----------------------+------------------+---------------------+------------------+-------------------------+--------------------+
|user_001|6735221ef265cb6c5eca0ce35f3502dfe2a10d9deea57eb0bb029627d6edd02f|7             |2026-07-28 09:51:34.380674|0.5               |1                      |4.3333335         |4.0              

In [42]:
from pyspark.sql.functions import (
    col,
    when,
    avg,
    row_number
)
from pyspark.sql.window import Window

latest_practice_window = (
    Window
    .partitionBy("user_id", "taxonomy_id")
    .orderBy(
        col("evidence_time").desc(),
        col("evidence_id").desc()
    )
)

latest_practice_df = (
    learner_concept_evidence_df
    .filter(col("evidence_type") == "practice_attempt")
    .withColumn(
        "practice_row_number",
        row_number().over(latest_practice_window)
    )
    .filter(col("practice_row_number") == 1)
    .select(
        "user_id",
        "taxonomy_id",
        col("score")
            .cast("float")
            .alias("last_practice_score")
    )
)

confusion_rate_df = (
    learner_concept_evidence_df
    .groupBy(
        "user_id",
        "taxonomy_id"
    )
    .agg(
        avg(
            when(
                col("still_confused").isNotNull(),
                col("still_confused").cast("int")
            )
        )
        .cast("float")
        .alias("confusion_rate")
    )
)

learner_concept_scoring_base_df = (
    learner_concept_components_df.alias("c")
    .join(
        latest_practice_df.alias("p"),
        ["user_id", "taxonomy_id"],
        "left"
    )
    .join(
        confusion_rate_df.alias("f"),
        ["user_id", "taxonomy_id"],
        "left"
    )
)

learner_concept_scoring_base_df.orderBy(
    "user_id",
    "taxonomy_id"
).show(truncate=False)

+--------+----------------------------------------------------------------+--------------+--------------------------+------------------+-----------------------+------------------+---------------------+------------------+-------------------------+--------------------+-------------------+--------------+
|user_id |taxonomy_id                                                     |evidence_count|last_evidence_time        |avg_practice_score|incorrect_attempt_count|avg_confidence_raw|avg_understanding_raw|avg_difficulty_raw|avg_validated_reliability|has_confusion_signal|last_practice_score|confusion_rate|
+--------+----------------------------------------------------------------+--------------+--------------------------+------------------+-----------------------+------------------+---------------------+------------------+-------------------------+--------------------+-------------------+--------------+
|user_001|6735221ef265cb6c5eca0ce35f3502dfe2a10d9deea57eb0bb029627d6edd02f|7             |2

In [43]:
from pyspark.sql.functions import (
    col,
    when,
    lit,
    greatest,
    round as spark_round
)

scoring_normalized_df = (
    learner_concept_scoring_base_df

    # המרה מסולם 1–10 לסולם 0–1
    .withColumn(
        "confidence_normalized",
        when(
            col("avg_confidence_raw").isNotNull(),
            col("avg_confidence_raw") / 10.0
        )
    )
    .withColumn(
        "understanding_normalized",
        when(
            col("avg_understanding_raw").isNotNull(),
            col("avg_understanding_raw") / 10.0
        )
    )
    .withColumn(
        "self_reported_difficulty_normalized",
        when(
            col("avg_difficulty_raw").isNotNull(),
            col("avg_difficulty_raw") / 10.0
        )
    )
    .withColumn(
        "practice_failure_rate",
        when(
            col("avg_practice_score").isNotNull(),
            1.0 - col("avg_practice_score")
        )
    )
)

In [44]:
scoring_with_mastery_df = (
    scoring_normalized_df

    .withColumn(
        "mastery_weight_sum",

        when(
            col("avg_practice_score").isNotNull(),
            lit(0.5)
        ).otherwise(lit(0.0))

        +

        when(
            col("understanding_normalized").isNotNull(),
            lit(0.3)
        ).otherwise(lit(0.0))

        +

        when(
            col("confidence_normalized").isNotNull(),
            lit(0.2)
        ).otherwise(lit(0.0))
    )

    .withColumn(
        "mastery_weighted_sum",

        when(
            col("avg_practice_score").isNotNull(),
            col("avg_practice_score") * 0.5
        ).otherwise(lit(0.0))

        +

        when(
            col("understanding_normalized").isNotNull(),
            col("understanding_normalized") * 0.3
        ).otherwise(lit(0.0))

        +

        when(
            col("confidence_normalized").isNotNull(),
            col("confidence_normalized") * 0.2
        ).otherwise(lit(0.0))
    )

    .withColumn(
        "mastery_score",
        when(
            col("mastery_weight_sum") > 0,
            spark_round(
                col("mastery_weighted_sum")
                / col("mastery_weight_sum"),
                4
            )
        )
    )
)

In [45]:
learner_concept_scores_df = (
    scoring_with_mastery_df

    .withColumn(
        "difficulty_weight_sum",

        when(
            col("practice_failure_rate").isNotNull(),
            lit(0.5)
        ).otherwise(lit(0.0))

        +

        when(
            col("self_reported_difficulty_normalized").isNotNull(),
            lit(0.3)
        ).otherwise(lit(0.0))

        +

        when(
            col("confusion_rate").isNotNull(),
            lit(0.2)
        ).otherwise(lit(0.0))
    )

    .withColumn(
        "difficulty_weighted_sum",

        when(
            col("practice_failure_rate").isNotNull(),
            col("practice_failure_rate") * 0.5
        ).otherwise(lit(0.0))

        +

        when(
            col("self_reported_difficulty_normalized").isNotNull(),
            col("self_reported_difficulty_normalized") * 0.3
        ).otherwise(lit(0.0))

        +

        when(
            col("confusion_rate").isNotNull(),
            col("confusion_rate") * 0.2
        ).otherwise(lit(0.0))
    )

    .withColumn(
        "difficulty_score",
        when(
            col("difficulty_weight_sum") > 0,
            spark_round(
                col("difficulty_weighted_sum")
                / col("difficulty_weight_sum"),
                4
            )
        )
    )

    .withColumn(
        "confidence_score",
        spark_round(
            col("confidence_normalized"),
            4
        )
    )

    .withColumn(
        "repeated_mistake_count",
        greatest(
            col("incorrect_attempt_count") - 1,
            lit(0)
        ).cast("int")
    )

    # סיכון משלב קושי גבוה ושליטה נמוכה
    .withColumn(
        "struggle_risk_score",
        when(
            col("difficulty_score").isNotNull()
            & col("mastery_score").isNotNull(),

            spark_round(
                col("difficulty_score") * 0.6
                + (1.0 - col("mastery_score")) * 0.4,
                4
            )
        )
        .when(
            col("difficulty_score").isNotNull(),
            col("difficulty_score")
        )
        .when(
            col("mastery_score").isNotNull(),
            spark_round(
                1.0 - col("mastery_score"),
                4
            )
        )
    )
)

In [46]:
learner_concept_scores_df.select(
    "user_id",
    "taxonomy_id",
    "evidence_count",
    "avg_practice_score",
    "confidence_score",
    "mastery_score",
    "difficulty_score",
    "repeated_mistake_count",
    "last_practice_score",
    "confusion_rate",
    "struggle_risk_score"
).orderBy(
    "user_id",
    "taxonomy_id"
).show(truncate=False)

+--------+----------------------------------------------------------------+--------------+------------------+----------------+-------------+----------------+----------------------+-------------------+--------------+-------------------+
|user_id |taxonomy_id                                                     |evidence_count|avg_practice_score|confidence_score|mastery_score|difficulty_score|repeated_mistake_count|last_practice_score|confusion_rate|struggle_risk_score|
+--------+----------------------------------------------------------------+--------------+------------------+----------------+-------------+----------------+----------------------+-------------------+--------------+-------------------+
|user_001|6735221ef265cb6c5eca0ce35f3502dfe2a10d9deea57eb0bb029627d6edd02f|7             |0.5               |0.4333          |0.4567       |0.675           |0                     |1.0                |1.0           |0.6223             |
|user_001|74aeea96e4ef671a43f520e045210290ff021dbd20708f

In [47]:
from pyspark.sql.functions import (
    col,
    lit,
    sha2,
    concat_ws
)

fact_learner_concept_state_df = (
    learner_concept_scores_df.alias("s")

    .join(
        dim_learner_df
        .select("user_id", "user_key")
        .alias("l"),
        col("s.user_id") == col("l.user_id"),
        "inner"
    )

    .join(
        dim_topic_df
        .select(
            col("taxonomy_id"),
            col("topic_key")
        )
        .alias("t"),
        col("s.taxonomy_id") == col("t.taxonomy_id"),
        "inner"
    )

    .select(
        sha2(
            concat_ws(
                "||",
                col("s.user_id"),
                col("s.taxonomy_id"),
                lit("1")
            ),
            256
        ).alias("concept_state_key"),

        col("l.user_key").cast("int"),
        col("t.topic_key").cast("int"),

        lit(1).cast("int").alias("state_version"),

        col("s.mastery_score").cast("float"),
        col("s.difficulty_score").cast("float"),
        col("s.confidence_score").cast("float"),

        col("s.repeated_mistake_count").cast("int"),
        col("s.last_practice_score").cast("float"),
        col("s.struggle_risk_score").cast("float"),

        col("s.evidence_count").cast("int"),
        col("s.last_evidence_time"),

        lit("prototype_weighted_v1")
            .alias("state_calculation_version"),

        col("s.last_evidence_time")
            .alias("valid_from"),

        lit(None)
            .cast("timestamp")
            .alias("valid_to"),

        lit(True)
            .cast("boolean")
            .alias("is_current")
    )
)

fact_learner_concept_state_df.orderBy(
    "user_key",
    "topic_key"
).show(truncate=False)

+----------------------------------------------------------------+--------+---------+-------------+-------------+----------------+----------------+----------------------+-------------------+-------------------+--------------+--------------------------+-------------------------+--------------------------+--------+----------+
|concept_state_key                                               |user_key|topic_key|state_version|mastery_score|difficulty_score|confidence_score|repeated_mistake_count|last_practice_score|struggle_risk_score|evidence_count|last_evidence_time        |state_calculation_version|valid_from                |valid_to|is_current|
+----------------------------------------------------------------+--------+---------+-------------+-------------+----------------+----------------+----------------------+-------------------+-------------------+--------------+--------------------------+-------------------------+--------------------------+--------+----------+
|974346c786ee1ba3c547c

In [48]:
from pyspark.sql.functions import col

invalid_concept_state_links_df = (
    fact_learner_concept_state_df
    .filter(
        col("concept_state_key").isNull()
        | col("user_key").isNull()
        | col("topic_key").isNull()
        | col("state_version").isNull()
        | col("last_evidence_time").isNull()
    )
)

invalid_concept_state_links_df.show(truncate=False)

print(
    "Rows with missing required links:",
    invalid_concept_state_links_df.count()
)

print(
    "Total concept-state rows:",
    fact_learner_concept_state_df.count()
)

print(
    "Distinct concept-state keys:",
    fact_learner_concept_state_df
    .select("concept_state_key")
    .distinct()
    .count()
)

+-----------------+--------+---------+-------------+-------------+----------------+----------------+----------------------+-------------------+-------------------+--------------+------------------+-------------------------+----------+--------+----------+
|concept_state_key|user_key|topic_key|state_version|mastery_score|difficulty_score|confidence_score|repeated_mistake_count|last_practice_score|struggle_risk_score|evidence_count|last_evidence_time|state_calculation_version|valid_from|valid_to|is_current|
+-----------------+--------+---------+-------------+-------------+----------------+----------------+----------------------+-------------------+-------------------+--------------+------------------+-------------------------+----------+--------+----------+
+-----------------+--------+---------+-------------+-------------+----------------+----------------+----------------------+-------------------+-------------------+--------------+------------------+-------------------------+----------+-

In [49]:
spark.sql("""
DELETE FROM demo.gold.fact_learner_concept_state
""")

DataFrame[]

In [50]:
fact_learner_concept_state_df.writeTo(
    "demo.gold.fact_learner_concept_state"
).append()

In [51]:
spark.sql("""
SELECT
    user_key,
    topic_key,
    state_version,
    mastery_score,
    difficulty_score,
    confidence_score,
    repeated_mistake_count,
    last_practice_score,
    struggle_risk_score,
    evidence_count,
    last_evidence_time,
    state_calculation_version,
    is_current
FROM demo.gold.fact_learner_concept_state
ORDER BY user_key, topic_key
""").show(truncate=False)

+--------+---------+-------------+-------------+----------------+----------------+----------------------+-------------------+-------------------+--------------+--------------------------+-------------------------+----------+
|user_key|topic_key|state_version|mastery_score|difficulty_score|confidence_score|repeated_mistake_count|last_practice_score|struggle_risk_score|evidence_count|last_evidence_time        |state_calculation_version|is_current|
+--------+---------+-------------+-------------+----------------+----------------+----------------------+-------------------+-------------------+--------------+--------------------------+-------------------------+----------+
|1       |5        |1            |0.4567       |0.675           |0.4333          |0                     |1.0                |0.6223             |7             |2026-07-28 09:51:34.380674|prototype_weighted_v1    |true      |
|1       |6        |1            |NULL         |NULL            |NULL            |0                 

In [52]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT concept_state_key) AS distinct_state_keys,
    COUNT(DISTINCT CONCAT(
        CAST(user_key AS STRING),
        '||',
        CAST(topic_key AS STRING),
        '||',
        CAST(state_version AS STRING)
    )) AS distinct_learner_topic_versions
FROM demo.gold.fact_learner_concept_state
""").show()

+----------+-------------------+-------------------------------+
|total_rows|distinct_state_keys|distinct_learner_topic_versions|
+----------+-------------------+-------------------------------+
|        12|                 12|                             12|
+----------+-------------------+-------------------------------+

